# Expert Sierra: Raw → Bronze load

Loads the Expert Sierra Raw tables into Bronze, 1:1. Every table goes through the shared `bronze_standard`
transformation (`config/bronze_transformations.yml`): string cleanup (trim, empty to NULL, control characters
removed, CHAR to STRING), then the audit columns `source_system`, `ingestion_timestamp`, `job_run_id` and
`record_hash` (empty for now).

**Settings** come from `config/bronze_load.yml`, with environment values (catalog) from `config/env/<env>.yaml`
at the repo root. **Which tables** to load, and how, is in `config/bronze_tables.yml`. Change those files, not this
notebook.

**No parameters.** The framework finds the environment by looking up this workspace's URL in
`config/env/environments.yaml` (`ConfigUtils.env_config_file()`); `job_run_id` is the job run id when run by
a job, `manual` otherwise.

**Load modes** (`load_mode` per table in `config/bronze_tables.yml`):
- `full`: overwrite Bronze from the current Raw snapshot. Use while Raw is truncated and reloaded.
- `incremental`: append new and updated Raw rows from the change feed, one checkpoint per table.
  Clear old checkpoints when switching to it.

In [ ]:
# Framework wheel, built with `uv build --wheel` and uploaded to the shared libraries folder
%pip install /Workspace/Shared/libraries/wilsonelser-dtf/wilsonelser_dtf-1.2.0-py3-none-any.whl

In [ ]:
%restart_python

In [ ]:
import json


def current_job_run_id() -> str:
    """The job run id when this notebook runs as a job task; "manual" for interactive runs."""
    try:
        context = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson())
        attributes = context.get("attributes", {})
        return str(attributes.get("multitaskParentRunId") or attributes.get("jobRunId") or "manual")
    except Exception:
        return "manual"


job_run_id = current_job_run_id()

In [ ]:
import json
import os
import sys
from pathlib import Path

# Config paths and custom functions are relative to the repo root (this notebook is in use_cases/expert_sierra)
repo_root = str(Path.cwd().parents[1])
if repo_root not in sys.path:
    sys.path.append(repo_root)

from pyspark.sql import functions as F

from wilsonelser.transformation.engine import TransformationEngine
from wilsonelser.transformation.utils.config_utils import ConfigUtils

# Environment config for this workspace, e.g. config/env/dev.yaml (from config/env/environments.yaml)
env_config_file = ConfigUtils.env_config_file()
settings = ConfigUtils.load_config("use_cases/expert_sierra/config/bronze_load.yml", env_config_file)["bronze_load"]

catalog = settings["catalog"]
raw_schema = settings["raw_schema"]
bronze_schema = settings["bronze_schema"]
CHANGE_FEED_COLUMNS = ["_change_type", "_commit_version", "_commit_timestamp"]

# Fills the ${job_run_id} placeholder in the transformation config
os.environ["job_run_id"] = job_run_id
transformer = TransformationEngine(settings["transformation_config"])

print(f"env config={env_config_file} catalog={catalog} job_run_id={job_run_id}")

In [ ]:
def tables_to_load() -> list:
    """The enabled tables from the tables config, as (name, load_mode) pairs."""
    tables = ConfigUtils.load_config(settings["tables_config"])["tables"]
    invalid = [t["name"] for t in tables if t.get("load_mode") not in ("full", "incremental")]
    if invalid:
        raise ValueError(f"load_mode must be full or incremental for: {', '.join(invalid)}")
    return [(t["name"], t["load_mode"]) for t in tables if t.get("enabled", True)]


def load_full(table: str) -> int:
    """Overwrites Bronze from the current Raw snapshot; returns the rows written."""
    bronze_table = f"{catalog}.{bronze_schema}.{table}"
    raw_df = spark.read.table(f"{catalog}.{raw_schema}.{table}")
    bronze_df = transformer.apply_transformations(settings["transformation_id"], raw_df)
    bronze_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)
    return spark.table(bronze_table).count()


def load_incremental(table: str) -> int:
    """Appends new and updated Raw rows from the change feed; returns the rows read.

    Raw tables are SCD Type 1 streaming tables, so updates rewrite rows and a plain readStream would fail
    on the first update. The change feed turns inserts and updates into an append-only stream.
    """
    raw_changes = (
        spark.readStream.option("readChangeFeed", "true")
        .table(f"{catalog}.{raw_schema}.{table}")
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
        .drop(*CHANGE_FEED_COLUMNS)
    )
    query = (
        transformer.apply_transformations(settings["transformation_id"], raw_changes)
        .writeStream.queryName(f"bronze_{table}")
        .option("checkpointLocation", f"{settings['checkpoint_location']}/{table}")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.{bronze_schema}.{table}")
    )
    query.awaitTermination()
    return sum(progress["numInputRows"] for progress in query.recentProgress)

In [ ]:
loaders = {"full": load_full, "incremental": load_incremental}
results = []
for table, load_mode in tables_to_load():
    try:
        rows = loaders[load_mode](table)
        results.append((table, load_mode, "loaded", rows, None))
    except Exception as error:
        results.append((table, load_mode, "failed", None, str(error)[:1000]))

summary = spark.createDataFrame(results, "table string, load_mode string, status string, rows long, error string")
display(summary)

In [ ]:
failed = [table for table, _, status, _, _ in results if status == "failed"]
if failed:
    raise RuntimeError(f"Bronze load failed for {len(failed)} table(s): {', '.join(failed)}")